# ATP Tennis – Logistic Regression Model
**19 player-specific features · train ≤ 2023 · val 2024 · test 2025**

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, accuracy_score
from sklearn.model_selection import PredefinedSplit, GridSearchCV

sns.set_theme(style='darkgrid')
_cwd = Path(os.getcwd())
ROOT = _cwd if (_cwd/'data').exists() else _cwd.parent
REPORTS = ROOT/'reports'; REPORTS.mkdir(exist_ok=True)
print(f'Root: {ROOT}')

## 1 · Load & Filter

In [ ]:
df = pd.read_parquet(ROOT/'data/processed/model_training_rows.parquet')
df['tourney_date_dt'] = pd.to_datetime(df['tourney_date_dt'])
df = df.query('a_pre_elo_matches_played >= 20 and b_pre_elo_matches_played >= 20').copy()
print(f'Rows: {len(df):,}  |  {df.tourney_date_dt.min().date()} → {df.tourney_date_dt.max().date()}')

## 2 · Player-Specific Features Only

In [ ]:
TARGET = 'player_a_wins'
FEATURES = [
    # Elo
    'diff_pre_surface_elo', 'diff_pre_overall_elo', 'elo_expected_a',
    # Rankings
    'diff_player_rank_points', 'diff_player_age',
    # Recent form
    'diff_pre_win_pct_last_10', 'diff_pre_surface_win_pct_last_10',
    # H2H
    'diff_pre_h2h_decayed_net',
    # Serve / return (non-collinear pair)
    'diff_pre_service_points_won_pct', 'diff_pre_return_points_won_pct',
    # Clutch
    'diff_pre_win_after_winning_first_set_pct', 'diff_pre_win_after_losing_first_set_pct',
    # Fatigue
    'diff_pre_fatigue_index_v1', 'diff_pre_days_since_last_match',
    'diff_pre_matches_last_7_days', 'diff_pre_minutes_last_7_days',
    # Injury proxy
    'diff_pre_retirements_last_10_matches',
    # Handedness
    'a_is_lefty', 'b_is_lefty',
]

dates = df['tourney_date_dt']
train_mask = dates.dt.year <= 2023
val_mask   = dates.dt.year == 2024
test_mask  = dates.dt.year == 2025

X_train=df.loc[train_mask,FEATURES]; y_train=df.loc[train_mask,TARGET].astype(int)
X_val  =df.loc[val_mask,  FEATURES]; y_val  =df.loc[val_mask,  TARGET].astype(int)
X_test =df.loc[test_mask, FEATURES]; y_test =df.loc[test_mask, TARGET].astype(int)
elo_prob_val =df.loc[val_mask,  'elo_expected_a'].values
elo_prob_test=df.loc[test_mask, 'elo_expected_a'].values
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,} | Features: {len(FEATURES)}')

## 3 · C Tuning (~5 sec)

In [ ]:
pipe = Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                 ('clf',LogisticRegression(max_iter=1000,solver='lbfgs'))])
fold_ids = np.full(len(X_train),-1)
for fi,vy in enumerate([2021,2022,2023]):
    fold_ids[(dates[train_mask].dt.year==vy).values]=fi

grid=GridSearchCV(pipe,{'clf__C':[0.01,0.05,0.1,0.5,1.0,2.0,5.0]},
                  cv=PredefinedSplit(fold_ids),scoring='neg_log_loss',n_jobs=-1)
grid.fit(X_train,y_train)
best_C=grid.best_params_['clf__C']
print(f'Best C={best_C}  |  CV log loss={-grid.best_score_:.4f}')
cv_df=pd.DataFrame({'C':[p['clf__C'] for p in grid.cv_results_['params']],
                    'Log Loss':-grid.cv_results_['mean_test_score']})
display(cv_df)

## 4 · Sigmoid Calibration

In [ ]:
model=grid.best_estimator_
cal=CalibratedClassifierCV(FrozenEstimator(model),method='sigmoid')
cal.fit(X_val,y_val)
p_val_raw=model.predict_proba(X_val)[:,1]; p_val_cal=cal.predict_proba(X_val)[:,1]
p_test_raw=model.predict_proba(X_test)[:,1]; p_test_cal=cal.predict_proba(X_test)[:,1]
print('Done.')

## 5 · Metrics

In [ ]:
def score(y_true,proba,label):
    return {'Model':label,'Log Loss':round(log_loss(y_true,proba),4),
            'Brier':round(brier_score_loss(y_true,proba),4),
            'AUC':round(roc_auc_score(y_true,proba),4),
            'Accuracy':round(accuracy_score(y_true,proba>=0.5),4)}

val_df=df.loc[val_mask].copy(); test_df=df.loc[test_mask].copy()
vbk=val_df[(val_df.bookmaker_odds_player_a>1)&(val_df.bookmaker_odds_player_b>1)].index
tbk=test_df[(test_df.bookmaker_odds_player_a>1)&(test_df.bookmaker_odds_player_b>1)].index
bk_val=(1/val_df.loc[vbk,'bookmaker_odds_player_a']).values
bk_test=(1/test_df.loc[tbk,'bookmaker_odds_player_a']).values

print('=== VALIDATION 2024 ===')
display(pd.DataFrame([score(y_val,elo_prob_val,'Elo baseline'),
                      score(y_val.loc[vbk],bk_val,'Bookmaker'),
                      score(y_val,p_val_raw,'LogReg raw'),
                      score(y_val,p_val_cal,'LogReg+Sigmoid')]))
print('\n=== TEST 2025 ===')
display(pd.DataFrame([score(y_test,elo_prob_test,'Elo baseline'),
                      score(y_test.loc[tbk],bk_test,'Bookmaker'),
                      score(y_test,p_test_raw,'LogReg raw'),
                      score(y_test,p_test_cal,'LogReg+Sigmoid')]))

## 6 · C Grid Plot

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(cv_df['C'],cv_df['Log Loss'],marker='o',lw=2,color='steelblue')
ax.axvline(best_C,color='red',linestyle='--',label=f'Best C={best_C}')
ax.set_xscale('log'); ax.set(title='C vs CV Log Loss',xlabel='C',ylabel='Log Loss'); ax.legend()
plt.tight_layout(); plt.savefig(REPORTS/'01_c_grid.png',dpi=150,bbox_inches='tight'); plt.show()

## 7 · Calibration Curves

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Calibration Curves',fontsize=13,fontweight='bold')
for ax,(y_true,pd_,title) in zip(axes,[
    (y_val,{'Elo':elo_prob_val,'LogReg':p_val_raw,'LogReg+Sig':p_val_cal},'Validation 2024'),
    (y_test,{'Elo':elo_prob_test,'LogReg':p_test_raw,'LogReg+Sig':p_test_cal},'Test 2025'),
]):
    ax.plot([0,1],[0,1],'k--',lw=1.5,label='Perfect')
    for name,proba in pd_.items():
        fp,mp=calibration_curve(y_true,proba,n_bins=12,strategy='quantile')
        ax.plot(mp,fp,marker='o',ms=5,label=name)
    ax.set(title=title,xlabel='Mean Predicted Prob',ylabel='Fraction Positives'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(REPORTS/'02_calibration.png',dpi=150,bbox_inches='tight'); plt.show()

## 8 · Feature Coefficients

In [ ]:
clf=model.named_steps['clf']
coef_df=(pd.DataFrame({'feature':FEATURES,'coef':clf.coef_[0]})
         .assign(abs_coef=lambda x:x.coef.abs())
         .sort_values('abs_coef',ascending=False))
display(coef_df[['feature','coef']].reset_index(drop=True))

fig,ax=plt.subplots(figsize=(10,7))
colors=['steelblue' if c>0 else 'crimson' for c in coef_df.coef]
ax.barh(coef_df.feature[::-1],coef_df.coef[::-1],color=colors[::-1],alpha=0.85)
ax.axvline(0,color='black',lw=1)
ax.set_title('Feature Coefficients – 19 Player-Specific Features',fontsize=12,fontweight='bold')
ax.set_xlabel('Coefficient (blue=favours player_a)')
plt.tight_layout(); plt.savefig(REPORTS/'03_feature_coefs.png',dpi=150,bbox_inches='tight'); plt.show()

## 9 · Value Betting Backtest (Test 2025)

In [ ]:
test_bt=test_df.copy(); test_bt['p_model']=p_test_cal
test_bt=test_bt[(test_bt.bookmaker_odds_player_a>1)&(test_bt.bookmaker_odds_player_b>1)].dropna(
    subset=['bookmaker_odds_player_a','bookmaker_odds_player_b'])

thresholds=np.arange(0.00,0.16,0.01); rois,volumes=[],[]
for thr in thresholds:
    ea=test_bt.p_model-(1/test_bt.bookmaker_odds_player_a)
    eb=(1-test_bt.p_model)-(1/test_bt.bookmaker_odds_player_b)
    ba,bb=(ea>thr).astype(int),(eb>thr).astype(int)
    pla=np.where(test_bt.player_a_wins==1,test_bt.bookmaker_odds_player_a-1,-1)*ba
    plb=np.where(test_bt.player_a_wins==0,test_bt.bookmaker_odds_player_b-1,-1)*bb
    nb=ba.sum()+bb.sum()
    rois.append((pla.sum()+plb.sum())/nb*100 if nb>0 else 0); volumes.append(int(nb))

best_thr=thresholds[int(np.argmax(rois))]
print(f'Optimal edge: {best_thr:.0%} | ROI={max(rois):.2f}% | Bets={volumes[int(np.argmax(rois))]}')

fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Value Betting Backtest – Test 2025',fontsize=13,fontweight='bold')
axes[0].bar(thresholds*100,rois,width=0.8,color=['green' if r>=0 else 'red' for r in rois],alpha=0.8)
axes[0].axhline(0,color='black',lw=1.2); axes[0].set(xlabel='Edge (%)',ylabel='ROI (%)',title='ROI')
axes[1].plot(thresholds*100,volumes,marker='o',ms=4,color='steelblue')
axes[1].set(xlabel='Edge (%)',ylabel='Bets',title='Volume')
plt.tight_layout(); plt.savefig(REPORTS/'04_backtest_roi.png',dpi=150,bbox_inches='tight'); plt.show()

## 10 · Cumulative P&L

In [ ]:
ea=test_bt.p_model-(1/test_bt.bookmaker_odds_player_a)
eb=(1-test_bt.p_model)-(1/test_bt.bookmaker_odds_player_b)
ba,bb=(ea>best_thr).astype(int),(eb>best_thr).astype(int)
pla=np.where(test_bt.player_a_wins==1,test_bt.bookmaker_odds_player_a-1,-1)*ba
plb=np.where(test_bt.player_a_wins==0,test_bt.bookmaker_odds_player_b-1,-1)*bb
cumpl=pd.Series(pla+plb).reset_index(drop=True).cumsum()

fig,ax=plt.subplots(figsize=(12,5))
ax.fill_between(range(len(cumpl)),cumpl,0,where=(cumpl>=0),alpha=0.25,color='green',label='Profit')
ax.fill_between(range(len(cumpl)),cumpl,0,where=(cumpl<0),alpha=0.25,color='red',label='Loss')
ax.plot(cumpl,lw=1.8,color='navy'); ax.axhline(0,color='black',lw=1)
ax.set(title=f'Cumulative P&L – Test 2025 (edge≥{best_thr:.0%})',xlabel='Bet #',ylabel='Units')
ax.legend(); plt.tight_layout()
plt.savefig(REPORTS/'05_cumulative_pnl.png',dpi=150,bbox_inches='tight'); plt.show()